# March Machine Learning Mania 2026 - Canonical Run Artifact Report

> **Script-first authority (D001):** Model training is allowed **only** via `run_pipeline.py` and `03_lgbm_train.py`.
> This notebook is **canonical artifact analysis/reporting only** and must not train or persist models.


## Cell 1: Imports and Paths
Latest run directory is selected from `mania_pipeline/artifacts/runs`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RUNS_ROOT = Path('../artifacts/runs')
assert RUNS_ROOT.exists(), f'Runs root not found: {RUNS_ROOT.resolve()}'
run_dirs = [p for p in RUNS_ROOT.iterdir() if p.is_dir()]
assert run_dirs, f'No run directories found under {RUNS_ROOT.resolve()}'
run_dir = max(run_dirs, key=lambda d: d.stat().st_mtime_ns)
print('Using run:', run_dir.name)


## Cell 2: Load Canonical Outputs


In [ ]:
run_metadata = json.loads((run_dir / 'run_metadata.json').read_text(encoding='utf-8'))
eval_report = json.loads((run_dir / 'eval_report.json').read_text(encoding='utf-8'))

stage_outputs = run_metadata.get('stage_outputs', {})
train_payload = stage_outputs.get('train', {})
print('Run status:', run_metadata.get('status'))
print('Available stage outputs:', list(stage_outputs.keys()))
print('Train payload keys:', list(train_payload.keys()))


## Cell 3: Split Metrics Table (Men/Women)


In [ ]:
metrics_table = pd.DataFrame(eval_report.get('metrics_table', []))
assert not metrics_table.empty, 'metrics_table is empty in eval_report.json'
metrics_table


## Cell 4: Side-by-Side Test Summary


In [ ]:
side = eval_report.get('side_by_side_summary', {})
side_df = pd.DataFrame([side])
side_df


## Cell 5: Quick Visualization (Test Brier by Gender)


In [ ]:
test_rows = metrics_table[metrics_table['split'] == 'Test'][['gender', 'brier']].copy()
test_rows = test_rows.sort_values('gender')
ax = test_rows.plot(kind='bar', x='gender', y='brier', legend=False, color=['#1f77b4', '#ff7f0e'])
ax.set_title('Test Brier by Gender (Lower is Better)')
ax.set_ylabel('Brier Score')
ax.set_xlabel('Gender')
plt.tight_layout()
test_rows
